# Week 12: Generative AI, Transformers & Diffusion
## Notebook 08: Flow Matching & Rectified Flows (Straight Trajectories)

### 1. Conceptual Intuition & Theoretical Background
Traditional diffusion models (DDPM/DDIM) introduce stochastic Brownian motion that forces probability trajectories to curve. Curved trajectories require tiny integration steps in ODE solvers.

**Flow Matching (Continuous Normalizing Flows)** constructs continuous probability paths between a source distribution $p_0(x) = \mathcal{N}(0, I)$ and target data distribution $p_1(x_1)$ using **straight linear interpolation**:

$$x_t = (1 - t) x_0 + t x_1, \quad t \in [0, 1]$$

The time-derivative yields a constant target velocity vector field:
$$v_t(x_t \mid x_0, x_1) = \frac{d x_t}{dt} = x_1 - x_0$$

The **Conditional Flow Matching (CFM) Objective** is a simple mean squared error loss:
$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim U[0, 1], x_0 \sim p_0, x_1 \sim p_1}\left[ \left\| v_\theta(x_t, t) - (x_1 - x_0) \right\|^2 \right]$$

Because paths are straight lines, an ODE solver can traverse from noise to clean data in very few steps without path deviation.

---

### 2. Scratch Python Implementation: Flow Matching Regressor

In [ ]:
import numpy as np

class FlowMatchingVectorFieldScratch:
    def __init__(self, dim=2):
        self.dim = dim
        # Linear layer mapping [x_t, t] -> velocity (dim + 1 -> dim)
        np.random.seed(42)
        self.W = np.random.normal(0, 0.1, (dim + 1, dim))
        self.b = np.zeros(dim)
        
    def predict_velocity(self, x_t, t):
        # Concatenate time t as a feature
        t_col = np.full((len(x_t), 1), t)
        features = np.hstack([x_t, t_col])
        return np.dot(features, self.W) + self.b
        
    def train_step(self, x0_noise, x1_target, lr=0.01):
        batch_size = len(x0_noise)
        # Sample random time steps t ~ Uniform(0, 1)
        t = np.random.uniform(0.0, 1.0, size=(batch_size, 1))
        
        # Linear interpolation path
        x_t = (1.0 - t) * x0_noise + t * x1_target
        
        # Target velocity is constant direction vector: x1 - x0
        target_v = x1_target - x0_noise
        
        # Predict velocity
        features = np.hstack([x_t, t])
        pred_v = np.dot(features, self.W) + self.b
        
        # MSE Gradient
        error = pred_v - target_v
        grad_W = np.dot(features.T, error) / batch_size
        grad_b = np.mean(error, axis=0)
        
        self.W -= lr * grad_W
        self.b -= lr * grad_b
        
        loss = np.mean(error ** 2)
        return loss

# Synthetic Data: Map standard Gaussian noise to 2D target cluster
n_pts = 1000
x0 = np.random.normal(0, 1, (n_pts, 2))
x1 = np.random.normal(loc=[5.0, -5.0], scale=0.5, size=(n_pts, 2))

model = FlowMatchingVectorFieldScratch(dim=2)
for epoch in range(500):
    loss = model.train_step(x0, x1, lr=0.05)

# ODE Sampling with Euler method from t=0 to t=1 in just 5 steps!
n_test = 5
test_noise = np.random.normal(0, 1, (n_test, 2))
current_x = test_noise.copy()

steps = 5
dt = 1.0 / steps
for step in range(steps):
    t_val = step * dt
    v = model.predict_velocity(current_x, t_val)
    current_x += v * dt  # Euler integration

print(f"Final Flow Matching Training Loss: {loss:.4f}")
print("Sampled Points from Noise in 5 Euler Steps:")
print(np.round(current_x, 2))
print("Notice how points land accurately around target distribution [5.0, -5.0]!")

### 3. Step-by-Step Code Walkthrough
1. **Linear Path Formulation**: $x_t = (1 - t)x_0 + t x_1$ forms straight lines without Brownian curves.
2. **Constant Target Velocity**: Target vector field is simply $x_1 - x_0$.
3. **Euler Solver**: Straight trajectories enable large step sizes without solver divergence.

---

### 4. Industry Applications & Trade-offs
- **State-of-the-Art Diffusion (Stable Diffusion 3, Flux.1):** Modern foundation models have completely transitioned from DDPM to Rectified Flow Matching.
- **Trade-off:** Random pairings of $x_0$ and $x_1$ can cause trajectory crossing; solved by Optimal Transport Flow Matching (OT-CFM).

---

### 5. Top Interview Questions & Answers
**Q1: Why do Rectified Flows (Flow Matching) outperform DDPM diffusion models in inference speed?**
> **Answer:** DDPM paths are curved due to Gaussian noise injection at every intermediate step. Rectified flows learn straight paths where velocity is constant, allowing numerical ODE solvers to take massive time steps without accumulating truncation errors.

**Q2: What is the "Reflow" procedure in Rectified Flows?**
> **Answer:** Reflow takes trained model trajectories, pairs generated samples with their specific initial noise seeds, and retrains the model on these straightened paths, eliminating remaining trajectory crossings.